In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.architecture import EFFICIENTNET_B0, VIT_SMALL, with_architecture_results_dir
from src.baseline_config import BASELINE_CONFIG, build_training_config
from src.phase1.config import PHASE1_IMAGES_DIR, PHASE1_RUNS, PHASE1_TRAIN_CSV
from src.phase2.config import (
    PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
    PHASE2_CONFIDENCE_ONLY_THRESHOLD,
    PHASE2_FRAMES_DIR,
    PHASE2_FULL_TRAIN_CSV,
    PHASE2_TRAIN_CSV,
)
from src.phase2.experiment import phase2_confidence_only_csv_path
from src.phase3.config import PHASE3_RUNS
from utils.common import RESULTS_DIR
from utils.constants import VALIDATION_CSV, VALIDATION_IMAGES_DIR
from utils.metrics import collect_results_metrics, summarize_general_results_metrics


ARCHITECTURE_CONFIG = BASELINE_CONFIG
CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics.csv"
FORCE_RECOMPUTE = False

TEST_CSV = Path("test/external_test.csv")
TEST_IMAGES_DIR = Path("test/images_cropped")


In [2]:
@dataclass(frozen=True)
class ExperimentNode:
    key: str
    label: str
    phase: str
    x: float
    y_offset: float
    train_csv: Path
    images_dir: Path
    results_dirs: tuple[Path, ...]
    random_states: tuple[int, ...]
    parent: str | None = None


def phase1_results_dirs() -> tuple[Path, ...]:
    return tuple(Path(run["results_dir"]) for run in PHASE1_RUNS)


def phase1_random_states() -> tuple[int, ...]:
    return tuple(int(run["random_state"]) for run in PHASE1_RUNS)


def phase3_results_dirs(descriptor: str) -> tuple[Path, ...]:
    return tuple(Path("phase3") / descriptor / run["seed_name"] for run in PHASE3_RUNS)


def phase3_random_states() -> tuple[int, ...]:
    return tuple(int(run["random_state"]) for run in PHASE3_RUNS)


def seed_dirs(root: Path) -> tuple[Path, ...]:
    return tuple(root / f"seed_{idx}" for idx in range(1, 5))


SEED_RANDOM_STATES = (42, 123, 456, 789)
CONF060_CSV = phase2_confidence_only_csv_path(PHASE2_CONFIDENCE_ONLY_THRESHOLD)

NODES = [
    ExperimentNode(
        key="phase1",
        label="F1 base",
        phase="Fase 1",
        x=1.0,
        y_offset=0.00,
        train_csv=PHASE1_TRAIN_CSV,
        images_dir=PHASE1_IMAGES_DIR,
        results_dirs=phase1_results_dirs(),
        random_states=phase1_random_states(),
    ),
    ExperimentNode(
        key="train",
        label="train",
        phase="Fase 2",
        x=2.0,
        y_offset=-0.28,
        train_csv=PHASE2_FULL_TRAIN_CSV,
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase2/train")),
        random_states=SEED_RANDOM_STATES,
        parent="phase1",
    ),
    ExperimentNode(
        key="conf060",
        label="conf060",
        phase="Fase 2",
        x=2.0,
        y_offset=0.28,
        train_csv=CONF060_CSV,
        images_dir=PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase2/conf060")),
        random_states=SEED_RANDOM_STATES,
        parent="phase1",
    ),
    ExperimentNode(
        key="train_conf040",
        label="train_conf040",
        phase="Fase 3",
        x=3.0,
        y_offset=-0.30,
        train_csv=PHASE2_TRAIN_CSV,
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=seed_dirs(Path("phase2/train_conf040")),
        random_states=SEED_RANDOM_STATES,
        parent="train",
    ),
    ExperimentNode(
        key="train_conf040_dedup_p75_25",
        label="train_conf040 dedup p75/25",
        phase="Fase 3",
        x=3.0,
        y_offset=0.00,
        train_csv=Path("phase3/phase3_train_conf040_dedup_p75_25.csv"),
        images_dir=PHASE2_FRAMES_DIR,
        results_dirs=phase3_results_dirs("train_conf040_dedup_p75_25"),
        random_states=phase3_random_states(),
        parent="train_conf040",
    ),
    ExperimentNode(
        key="conf060_dedup_p75_25",
        label="conf060 dedup p75/25",
        phase="Fase 3",
        x=3.0,
        y_offset=0.30,
        train_csv=Path("phase3/phase3_conf060_dedup_p75_25.csv"),
        images_dir=PHASE2_CONFIDENCE_ONLY_FRAMES_DIR,
        results_dirs=phase3_results_dirs("conf060_dedup_p75_25"),
        random_states=phase3_random_states(),
        parent="conf060",
    ),
]

pd.DataFrame(
    [
        {
            "key": node.key,
            "phase": node.phase,
            "parent": node.parent,
            "train_csv": str(node.train_csv),
            "images_dir": str(node.images_dir),
            "results_dirs": ", ".join(str(path) for path in node.results_dirs),
        }
        for node in NODES
    ]
)


,key,phase,parent,train_csv,images_dir,results_dirs
0,phase1,Fase 1,NaN,phase1_train.csv,images_cropped,"phase1\seed1, phase1\seed2, phase1\seed3, phas..."
1,train,Fase 2,phase1,phase2\phase2_train.csv,phase2\frames,"phase2\train\seed_1, phase2\train\seed_2, phas..."
2,conf060,Fase 2,phase1,phase2\phase2_train_conf060.csv,phase2\frames_confidence_only,"phase2\conf060\seed_1, phase2\conf060\seed_2, ..."
3,train_conf040,Fase 3,train,phase2\phase2_train_conf040.csv,phase2\frames,"phase2\train_conf040\seed_1, phase2\train_conf..."
4,train_conf040_dedup_p75_25,Fase 3,train_conf040,phase3\phase3_train_conf040_dedup_p75_25.csv,phase2\frames,"phase3\train_conf040_dedup_p75_25\seed_1, phas..."
5,conf060_dedup_p75_25,Fase 3,conf060,phase3\phase3_conf060_dedup_p75_25.csv,phase2\frames_confidence_only,"phase3\conf060_dedup_p75_25\seed_1, phase3\con..."


In [ ]:
def summarize_node_split(node: ExperimentNode, split: str, csv_path: Path, images_dir: Path) -> dict:
    existing_results_dirs = [
        results_dir
        for results_dir in node.results_dirs
        if (
            RESULTS_DIR
            / with_architecture_results_dir(ARCHITECTURE_CONFIG.architecture, results_dir)
            / "best_baseline_model.pth"
        ).exists()
    ]
    if not existing_results_dirs:
        return {
            "key": node.key,
            "label": node.label,
            "phase": node.phase,
            "parent": node.parent,
            "split": split,
            "n_seeds": 0,
            "macro_f1": np.nan,
            "macro_f1_std": np.nan,
            "mcc": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
        }

    random_states = node.random_states[: len(existing_results_dirs)]
    per_seed = collect_results_metrics(
        results_dirs=existing_results_dirs,
        validation_csv_dir=csv_path,
        validation_img_dir=images_dir,
        training_config=ARCHITECTURE_CONFIG,
        random_states=random_states,
    )
    summary = summarize_general_results_metrics(per_seed)
    return {
        "key": node.key,
        "label": node.label,
        "phase": node.phase,
        "parent": node.parent,
        "split": split,
        "n_seeds": len(existing_results_dirs),
        "macro_f1": summary["macro_f1"],
        "macro_f1_std": summary["macro_f1_std"],
        "mcc": summary["mcc"],
        "accuracy": summary["accuracy"],
        "precision": summary["precision"],
        "recall": summary["recall"],
    }


def compute_lineage_metrics() -> pd.DataFrame:
    rows = []
    for node in NODES:
        rows.append(
            summarize_node_split(
                node=node,
                split="validation",
                csv_path=VALIDATION_CSV,
                images_dir=VALIDATION_IMAGES_DIR,
            )
        )
        rows.append(
            summarize_node_split(
                node=node,
                split="test",
                csv_path=TEST_CSV,
                images_dir=TEST_IMAGES_DIR,
            )
        )
    return pd.DataFrame(rows)


if CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    metrics_df = pd.read_csv(CACHE_PATH)
else:
    metrics_df = compute_lineage_metrics()
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    metrics_df.to_csv(CACHE_PATH, index=False)

metrics_df.sort_values(["split", "macro_f1"], ascending=[True, False])


In [ ]:
summary_table = (
    metrics_df
    .assign(parent=lambda df: df["parent"].fillna("-"))
    .pivot_table(
        index=["phase", "label", "parent", "n_seeds"],
        columns="split",
        values=["macro_f1", "macro_f1_std", "mcc", "accuracy", "precision", "recall"],
        aggfunc="first",
    )
    .sort_values(("macro_f1", "test"), ascending=False)
)

summary_table


In [ ]:
ARCHITECTURE_CONFIGS = {
    "EfficientNet": build_training_config(EFFICIENTNET_B0),
    "ViT-Small": build_training_config(VIT_SMALL),
}
MULTI_ARCH_CACHE_PATH = RESULTS_DIR / "final_lineage_validation_test_metrics_by_architecture.csv"


def summarize_node_split_for_architecture(
    node: ExperimentNode,
    split: str,
    csv_path: Path,
    images_dir: Path,
    architecture_label: str,
    architecture_config,
) -> dict:
    existing_results_dirs = [
        results_dir
        for results_dir in node.results_dirs
        if (
            RESULTS_DIR
            / with_architecture_results_dir(architecture_config.architecture, results_dir)
            / "best_baseline_model.pth"
        ).exists()
    ]
    if not existing_results_dirs:
        return {
            "architecture": architecture_label,
            "key": node.key,
            "label": node.label,
            "phase": node.phase,
            "parent": node.parent,
            "split": split,
            "n_seeds": 0,
            "macro_f1": np.nan,
            "macro_f1_std": np.nan,
            "mcc": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
        }

    random_states = node.random_states[: len(existing_results_dirs)]
    per_seed = collect_results_metrics(
        results_dirs=existing_results_dirs,
        validation_csv_dir=csv_path,
        validation_img_dir=images_dir,
        training_config=architecture_config,
        random_states=random_states,
    )
    summary = summarize_general_results_metrics(per_seed)
    return {
        "architecture": architecture_label,
        "key": node.key,
        "label": node.label,
        "phase": node.phase,
        "parent": node.parent,
        "split": split,
        "n_seeds": len(existing_results_dirs),
        "macro_f1": summary["macro_f1"],
        "macro_f1_std": summary["macro_f1_std"],
        "mcc": summary["mcc"],
        "accuracy": summary["accuracy"],
        "precision": summary["precision"],
        "recall": summary["recall"],
    }


def compute_multi_arch_metrics() -> pd.DataFrame:
    rows = []
    for architecture_label, architecture_config in ARCHITECTURE_CONFIGS.items():
        for node in NODES:
            rows.append(
                summarize_node_split_for_architecture(
                    node=node,
                    split="validation",
                    csv_path=VALIDATION_CSV,
                    images_dir=VALIDATION_IMAGES_DIR,
                    architecture_label=architecture_label,
                    architecture_config=architecture_config,
                )
            )
            rows.append(
                summarize_node_split_for_architecture(
                    node=node,
                    split="test",
                    csv_path=TEST_CSV,
                    images_dir=TEST_IMAGES_DIR,
                    architecture_label=architecture_label,
                    architecture_config=architecture_config,
                )
            )
    return pd.DataFrame(rows)


if MULTI_ARCH_CACHE_PATH.exists() and not FORCE_RECOMPUTE:
    multi_arch_metrics_df = pd.read_csv(MULTI_ARCH_CACHE_PATH)
else:
    multi_arch_metrics_df = compute_multi_arch_metrics()
    MULTI_ARCH_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    multi_arch_metrics_df.to_csv(MULTI_ARCH_CACHE_PATH, index=False)

multi_arch_metrics_df.sort_values(
    ["split", "architecture", "macro_f1"],
    ascending=[True, True, False],
)


In [ ]:
def plot_architecture_lineage(
    data: pd.DataFrame,
    architectures: tuple[str, ...],
    title: str,
    ax=None,
):
    node_by_key = {node.key: node for node in NODES}
    plot_nodes = [node for node in NODES]
    plot_parent_by_key = {}
    if ax is None:
        _, ax = plt.subplots(figsize=(13, 7))

    architecture_styles = {
        "EfficientNet": {"offset": -0.055, "marker": "o", "alpha": 0.80},
        "ViT-Small": {"offset": 0.055, "marker": "^", "alpha": 0.80},
    }
    split_styles = {
        "validation": {"color": "#2563eb", "linestyle": "--", "label": "Validation", "offset": -0.022, "label_y": -15},
        "test": {"color": "#dc2626", "linestyle": "-", "label": "Test", "offset": 0.022, "label_y": 11},
    }

    for architecture in architectures:
        arch_style = architecture_styles[architecture]
        architecture_df = data[data["architecture"].eq(architecture)]
        metric_by_key_split = {
            (row.key, row.split): row
            for row in architecture_df.itertuples(index=False)
        }

        for split, split_style in split_styles.items():
            legend_label = f"{architecture} - {split_style['label']}"
            for node_index, node in enumerate(plot_nodes):
                row = metric_by_key_split.get((node.key, split))
                if row is None or pd.isna(row.macro_f1):
                    continue
                x = node.x + node.y_offset + arch_style["offset"] + split_style["offset"]
                yerr = 0 if pd.isna(row.macro_f1_std) else row.macro_f1_std
                ax.errorbar(
                    x,
                    row.macro_f1,
                    yerr=yerr,
                    fmt=arch_style["marker"],
                    color=split_style["color"],
                    ecolor=split_style["color"],
                    elinewidth=1.2,
                    capsize=3,
                    markersize=7,
                    alpha=arch_style["alpha"],
                    label=legend_label if node_index == 0 else None,
                )
                ax.annotate(
                    f"{row.macro_f1:.2f}",
                    xy=(x, row.macro_f1),
                    xytext=(0, split_style["label_y"]),
                    textcoords="offset points",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=split_style["color"],
                )

            for node in plot_nodes:
                parent_key = plot_parent_by_key.get(node.key, node.parent)
                if parent_key is None:
                    continue
                parent = node_by_key[parent_key]
                parent_row = metric_by_key_split.get((parent.key, split))
                child_row = metric_by_key_split.get((node.key, split))
                if parent_row is None or child_row is None:
                    continue
                if pd.isna(parent_row.macro_f1) or pd.isna(child_row.macro_f1):
                    continue
                ax.plot(
                    [
                        parent.x + parent.y_offset + arch_style["offset"] + split_style["offset"],
                        node.x + node.y_offset + arch_style["offset"] + split_style["offset"],
                    ],
                    [parent_row.macro_f1, child_row.macro_f1],
                    color=split_style["color"],
                    linestyle=split_style["linestyle"],
                    linewidth=1.1,
                    alpha=0.35,
                )

    for node in plot_nodes:
        node_rows = data[data["key"].eq(node.key)]
        if node_rows["macro_f1"].dropna().empty:
            continue
        ax.annotate(
            node.label,
            xy=(node.x + node.y_offset, 0.985),
            xycoords=("data", "axes fraction"),
            xytext=(0, -4),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=8.5,
            rotation=30,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.75, "pad": 1.5},
            clip_on=False,
        )

    ax.set_xlim(0.65, 3.62)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Fase 1", "Fase 2", "Fase 3"])
    ax.set_ylabel("Macro-F1")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="lower right", fontsize=9)
    ax.margins(x=0.08)

    valid_f1 = data["macro_f1"].dropna()
    if not valid_f1.empty:
        margin = max(0.015, float(valid_f1.std()) * 0.75)
        ax.set_ylim(
            max(0, float(valid_f1.min()) - margin),
            min(1, float(valid_f1.max()) + margin * 1.8),
        )
    return ax


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6), sharey=True)

plot_architecture_lineage(
    multi_arch_metrics_df[multi_arch_metrics_df["architecture"].eq("EfficientNet")],
    architectures=("EfficientNet",),
    title="EfficientNet",
    ax=axes[0],
)
plot_architecture_lineage(
    multi_arch_metrics_df[multi_arch_metrics_df["architecture"].eq("ViT-Small")],
    architectures=("ViT-Small",),
    title="ViT-Small",
    ax=axes[1],
)

plt.tight_layout()
plt.show()


In [ ]:
multi_arch_summary_table = (
    multi_arch_metrics_df
    .assign(parent=lambda df: df["parent"].fillna("-"))
    .pivot_table(
        index=["architecture", "phase", "label", "parent", "n_seeds"],
        columns="split",
        values=["macro_f1", "macro_f1_std", "mcc", "accuracy", "precision", "recall"],
        aggfunc="first",
    )
    .sort_values([("macro_f1", "test"), ("macro_f1", "validation")], ascending=False)
)

multi_arch_summary_table
